# Offline preference-classification submission

Train a GRU and lexical/structural classifiers locally within the notebook, then blend
the models with weights and a temperature frozen on a prompt-grouped development set.
Refit uses every labeled training row. No external APIs or downloads are needed.

This is the deployable subset of the broader TF-IDF, recurrent and Qwen3 embedding
study. Model identities and IDs are excluded from features. Responses are augmented
by swapping A/B with corresponding labels; inference averages both orientations.
The program handles the three example rows or the full hidden test set supplied by Kaggle.


In [1]:
import os, sys, json
from pathlib import Path
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
INPUTS = [Path('/kaggle/input/llm-classification-finetuning'), Path('/kaggle/input/competitions/llm-classification-finetuning')]
if any(p.exists() for p in INPUTS):
    RAW = next(p for p in INPUTS if p.exists())
    WORK = Path('/kaggle/working')
else:
    RAW = Path.home() / 'database/llm_classification_finetuning/raw'
    WORK = Path.home() / 'database/llm_classification_finetuning/runs/full_refit'
WORK.mkdir(parents=True, exist_ok=True)
MODULES = WORK / 'bundled_code'
(MODULES / 'arena').mkdir(parents=True, exist_ok=True)
(MODULES / 'arena/__init__.py').write_text('')
sys.path.insert(0, str(MODULES))


Bundled source: `arena/data.py`.

In [2]:
(MODULES / 'arena/data.py').write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nROOT = Path(os.environ.get("ARENA_DATA_DIR", Path.home() / "database/llm_classification_finetuning"))\nCLASSES = ["winner_model_a", "winner_model_b", "winner_tie"]\nTEXT_COLUMNS = ["prompt", "response_a", "response_b"]\nSEED = 20261101\n\n\ndef parse_turns(value):\n    """Kaggle stores JSON arrays in CSV fields; preserve turn boundaries and nulls."""\n    if not isinstance(value, str):\n        return ""\n    try:\n        turns = json.loads(value)\n    except json.JSONDecodeError as exc:\n        raise ValueError("Expected a JSON-encoded conversation list") from exc\n    if not isinstance(turns, list):\n        raise ValueError("Expected a list of conversation turns")\n    return "\\n\\n[TURN]\\n\\n".join("" if turn is None else str(turn) for turn in turns)\n\n\ndef load_data():\n    train = pd.read_csv(ROOT / "raw/train.csv")\n    test = pd.read_csv(ROOT / "raw/test.csv")\n    for frame in (train, test):\n        for col in TEXT_COLUMNS:\n            frame[col] = frame[col].map(parse_turns)\n    assert train[CLASSES].isin([0, 1]).all().all()\n    assert train[CLASSES].sum(axis=1).eq(1).all()\n    assert train.id.is_unique and test.id.is_unique\n    return train, test\n\n\ndef prompt_groups(frame):\n    return frame.prompt.map(lambda s: hashlib.sha256(re.sub(r"\\s+", " ", s).strip().casefold().encode()).hexdigest())\n\n\ndef split_data(frame):\n    """8 train folds, 1 development fold, 1 untouched final holdout fold."""\n    y = frame[CLASSES].values.argmax(1)\n    groups = prompt_groups(frame)\n    fold = np.full(len(frame), -1)\n    cv = StratifiedGroupKFold(10, shuffle=True, random_state=SEED)\n    for k, (_, idx) in enumerate(cv.split(frame, y, groups)):\n        fold[idx] = k\n    parts = {"train": np.flatnonzero(fold >= 2), "dev": np.flatnonzero(fold == 1), "holdout": np.flatnonzero(fold == 0)}\n    for a, b in [("train", "dev"), ("train", "holdout"), ("dev", "holdout")]:\n        assert not set(groups.iloc[parts[a]]) & set(groups.iloc[parts[b]])\n    return y, parts, fold\n\n\ndef structural_features(frame):\n    """Only features available in test. Never use model identities or IDs."""\n    values = []\n    for col in TEXT_COLUMNS:\n        text = frame[col]\n        n = text.str.len().to_numpy()\n        words = text.str.count(r"\\S+").to_numpy()\n        values.append(np.column_stack([\n            np.log1p(n), np.log1p(words), np.log1p(text.str.count("\\n")),\n            np.log1p(text.str.count("```")), np.log1p(text.str.count(r"[!?]")),\n            np.log1p(text.str.count(r"(?m)^\\s*(?:[-*]|\\d+\\.)\\s")),\n            text.str.contains(r"(?i)\\b(?:sorry|cannot|can\'t|unable|as an ai)\\b", regex=True).astype(float),\n            text.str.count(r"[^\\x00-\\x7F]").to_numpy() / np.maximum(n, 1),\n            (words == 0).astype(float),\n        ]))\n    p, a, b = values\n    return np.column_stack([p, a, b, a-b, np.abs(a-b)]).astype(np.float32)\n\n\ndef swapped(frame):\n    out = frame.copy()\n    out[["response_a", "response_b"]] = frame[["response_b", "response_a"]].values\n    return out\n\n\ndef dense_pair_features(embeddings, stats):\n    p, a, b = embeddings\n    return np.column_stack([p, a, b, a-b, np.abs(a-b), a*b, p*a, p*b, stats]).astype(np.float32)\n\n\ndef swapped_pair_features(embeddings, stats):\n    p, a, b = embeddings\n    d = stats.shape[1] // 5\n    sp, sa, sb, delta, absolute = np.split(stats, [d, 2*d, 3*d, 4*d], axis=1)\n    return dense_pair_features((p, b, a), np.column_stack([sp, sb, sa, -delta, absolute]))\n\n\ndef json_write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, default=lambda x: x.item() if hasattr(x, "item") else str(x)) + "\\n")\n    temp.replace(path)\n')
print('Bundled arena/data.py')

Bundled arena/data.py


Bundled source: `arena/models.py`.

In [3]:
(MODULES / 'arena/models.py').write_text('from __future__ import annotations\n\nimport copy\nimport time\n\nimport joblib\nimport lightgbm as lgb\nimport numpy as np\nimport torch\nfrom scipy import sparse\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss\nfrom sklearn.preprocessing import StandardScaler\nfrom torch import nn\n\nfrom .data import SEED, json_write\n\n\ndef scores(y,p):\n    assert p.shape==(len(y),3) and np.isfinite(p).all()\n    assert np.all(p>=0) and np.allclose(p.sum(1),1,atol=1e-5)\n    return {"log_loss":float(log_loss(y,p,labels=[0,1,2])),\n            "accuracy":float(accuracy_score(y,p.argmax(1))),\n            "balanced_accuracy":float(balanced_accuracy_score(y,p.argmax(1))),\n            "macro_f1":float(f1_score(y,p.argmax(1),average="macro",zero_division=0)),\n            "brier":float(np.mean(np.sum((p-np.eye(3)[y])**2,axis=1)))}\n\n\ndef symmetric_predict(model,x,xswap):\n    return (model.predict_proba(x)+model.predict_proba(xswap)[:,[1,0,2]])*.5\n\n\ndef augment(x,xswap,y,idx):\n    if sparse.issparse(x):\n        xx=sparse.vstack([x[idx],xswap[idx]],format="csr")\n    else:\n        xx=np.concatenate([x[idx],xswap[idx]])\n    return xx,np.concatenate([y[idx],np.array([1,0,2])[y[idx]]])\n\n\nclass NeuralClassifier:\n    def __init__(self,input_dim):\n        torch.manual_seed(SEED)\n        self.model=nn.Sequential(nn.Linear(input_dim,128),nn.ReLU(),nn.Dropout(.4),nn.Linear(128,3))\n        self.scaler=StandardScaler()\n        self.history=[]\n\n    def fit(self,x,y,dev,devswap,dev_y,epochs=30):\n        torch.set_num_threads(4)\n        xx=torch.as_tensor(self.scaler.fit_transform(x).astype(np.float32))\n        yy=torch.as_tensor(y,dtype=torch.long)\n        optimizer=torch.optim.AdamW(self.model.parameters(),lr=.0005,weight_decay=.05)\n        best=100.;stale=0;state=None\n        generator=torch.Generator().manual_seed(SEED)\n        for epoch in range(epochs):\n            self.model.train()\n            order=torch.randperm(len(y),generator=generator)\n            for batch in order.split(256):\n                optimizer.zero_grad(set_to_none=True)\n                loss=nn.functional.cross_entropy(self.model(xx[batch]),yy[batch],label_smoothing=.02)\n                loss.backward();optimizer.step()\n            p=symmetric_predict(self,dev,devswap)\n            value=log_loss(dev_y,p,labels=[0,1,2])\n            self.history.append({"epoch":epoch+1,"dev_log_loss":value})\n            if value<best-.0001:\n                best=value;state=copy.deepcopy(self.model.state_dict());stale=0\n            else:\n                stale+=1\n                if stale>=4:break\n        self.model.load_state_dict(state)\n        return self\n\n    @torch.inference_mode()\n    def predict_proba(self,x):\n        self.model.eval()\n        result=[]\n        for start in range(0,len(x),1024):\n            xx=torch.as_tensor(self.scaler.transform(x[start:start+1024]).astype(np.float32))\n            result.append(self.model(xx).softmax(1).numpy())\n        return np.concatenate(result)\n\n\ndef fit_classifiers(name,x,xswap,y,parts,out,kinds=("logistic","gbdt","neural")):\n    """Select settings on dev; do not read holdout labels here."""\n    out.mkdir(parents=True,exist_ok=True)\n    xx,yy=augment(x,xswap,y,parts["train"])\n    dev=parts["dev"]\n    results=[]\n    for kind in kinds:\n        started=time.monotonic();settings={}\n        if kind=="logistic":\n            # Sparse TF-IDF is naturally scaled. Dense vectors include structural features.\n            scaler=StandardScaler(with_mean=not sparse.issparse(x),with_std=not sparse.issparse(x)).fit(xx)\n            sx=scaler.transform(xx)\n            best=100.;trials=[];model=None\n            for c in ([.1,1.,3.] if sparse.issparse(x) else [.001,.01,.1]):\n                candidate=LogisticRegression(C=c,max_iter=250,solver="lbfgs",random_state=SEED)\n                candidate.fit(sx,yy)\n                p=(candidate.predict_proba(scaler.transform(x[dev]))+candidate.predict_proba(scaler.transform(xswap[dev]))[:,[1,0,2]])*.5\n                loss=log_loss(y[dev],p,labels=[0,1,2]);trials.append({"C":c,"dev_log_loss":loss})\n                if loss<best:best=loss;model=candidate;settings={"C":c}\n            from sklearn.pipeline import make_pipeline\n            model=make_pipeline(scaler,model)\n            settings["trials"]=trials\n        elif kind=="gbdt":\n            model=lgb.LGBMClassifier(n_estimators=1500,learning_rate=.03,num_leaves=15,\n                min_child_samples=100,colsample_bytree=.8,subsample=.8,subsample_freq=1,\n                reg_lambda=5.,verbosity=-1,n_jobs=4,random_state=SEED,deterministic=True,force_col_wise=True)\n            model.fit(xx,yy,eval_set=[(x[dev],y[dev]),(xswap[dev],np.array([1,0,2])[y[dev]])],\n                callbacks=[lgb.early_stopping(60,verbose=False)])\n            settings={"best_iteration":model.best_iteration_,"num_leaves":15}\n        elif kind=="neural":\n            model=NeuralClassifier(x.shape[1]).fit(xx,yy,x[dev],xswap[dev],y[dev])\n            settings={"history":model.history,"epochs":len(model.history)}\n        else:raise ValueError(kind)\n        # Save predictions for all rows; metrics for holdout are computed only at the final frozen comparison.\n        p=symmetric_predict(model,x,xswap)\n        model_name=f"{name}__{kind}"\n        np.save(out/f"{model_name}_predictions.npy",p)\n        joblib.dump(model,out/f"{model_name}.joblib")\n        row={"model":model_name,"representation":name,"classifier":kind,\n             "dev":scores(y[dev],p[dev]),"seconds":time.monotonic()-started,"settings":settings,\n             "n_train":len(parts["train"]),"feature_count":x.shape[1]}\n        json_write(out/f"{model_name}.json",row)\n        results.append(row)\n        print(model_name,row["dev"],f\'{row["seconds"]:.1f}s\',flush=True)\n    return results\n')
print('Bundled arena/models.py')

Bundled arena/models.py


Bundled source: `arena/rnn.py`.

In [4]:
(MODULES / 'arena/rnn.py').write_text('from __future__ import annotations\n\nimport copy\nimport re\nimport time\nfrom collections import Counter\n\nimport numpy as np\nimport torch\nfrom sklearn.metrics import log_loss\nfrom sklearn.preprocessing import StandardScaler\nfrom torch import nn\n\nfrom .data import SEED, TEXT_COLUMNS, json_write\n\nTOKEN = re.compile(r"\\w+|[^\\w\\s]", re.UNICODE)\n\n\ndef tokenize(text):\n    return TOKEN.findall(text.casefold())\n\n\ndef make_sequences(frame, train_idx, max_tokens=128, vocab_size=30000):\n    counts = Counter()\n    for col in TEXT_COLUMNS:\n        for text in frame.iloc[train_idx][col]:\n            counts.update(tokenize(text))\n    vocab = {word:i+2 for i,(word,count) in enumerate(counts.most_common(vocab_size-2)) if count>=2}\n    arrays=[]\n    diagnostics={}\n    for col in TEXT_COLUMNS:\n        array=np.zeros((len(frame),max_tokens),dtype=np.int32)\n        truncated=unknown=total=0\n        for i,text in enumerate(frame[col]):\n            words=tokenize(text)\n            truncated+=len(words)>max_tokens\n            if len(words)>max_tokens: words=words[:max_tokens//2]+words[-max_tokens//2:]\n            ids=[vocab.get(w,1) for w in words] or [1]\n            array[i,:len(ids)]=ids\n            unknown+=sum(t==1 for t in ids);total+=len(ids)\n        arrays.append(array)\n        diagnostics[col]={"truncation_rate":truncated/len(frame),"unknown_token_rate":unknown/total}\n    return np.stack(arrays,axis=1),vocab,diagnostics\n\n\nclass PreferenceRNN(nn.Module):\n    """Shared GRU, learned from this competition\'s training texts and labels."""\n    def __init__(self,vocab_size,stats_dim,embedding_dim=64,hidden=48):\n        super().__init__()\n        self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx=0)\n        self.gru=nn.GRU(embedding_dim,hidden,batch_first=True)\n        self.dropout=nn.Dropout(.2)\n        self.head=nn.Sequential(nn.Linear(8*hidden*2+stats_dim,128),nn.ReLU(),nn.Dropout(.35),nn.Linear(128,3))\n\n    def encode(self,ids):\n        mask=ids.ne(0)\n        outputs,_=self.gru(self.dropout(self.embedding(ids)))\n        lengths=mask.sum(1).clamp(min=1)\n        mean=(outputs*mask.unsqueeze(-1)).sum(1)/lengths[:,None]\n        last=outputs[torch.arange(len(ids)),lengths-1]\n        return torch.cat([mean,last],dim=1)\n\n    def forward(self,seq,stats):\n        n,fields,length=seq.shape\n        enc=self.encode(seq.reshape(n*fields,length)).reshape(n,fields,-1)\n        p,a,b=enc.unbind(1)\n        features=torch.cat([p,a,b,a-b,torch.abs(a-b),a*b,p*a,p*b,stats],dim=1)\n        return self.head(features)\n\n\ndef train_rnn(frame,labels,parts,stats,out,epochs=12,max_tokens=128,hidden=48):\n    torch.manual_seed(SEED)\n    np.random.seed(SEED)\n    torch.set_num_threads(4)\n    out.mkdir(parents=True,exist_ok=True)\n    seq,vocab,diagnostics=make_sequences(frame,parts["train"],max_tokens=max_tokens)\n    scaler=StandardScaler().fit(stats[parts["train"]])\n    scaled=scaler.transform(stats).astype(np.float32)\n    # Standardize after swapping raw structural features.\n    swapped_stats=np.concatenate([stats[:,:9],stats[:,18:27],stats[:,9:18],-stats[:,27:36],stats[:,36:45]],axis=1)\n    scaled_swap=scaler.transform(swapped_stats).astype(np.float32)\n    model=PreferenceRNN(len(vocab)+2,stats.shape[1],hidden=hidden)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)\n    loss_fn=nn.CrossEntropyLoss(label_smoothing=.02)\n    rng=np.random.default_rng(SEED)\n    best=100.;stale=0;history=[];best_state=None\n\n    @torch.inference_mode()\n    def predict(idx):\n        model.eval();result=[]\n        for start in range(0,len(idx),256):\n            rows=idx[start:start+256]\n            x=torch.as_tensor(seq[rows],dtype=torch.long)\n            s=torch.as_tensor(scaled[rows])\n            ss=torch.as_tensor(scaled_swap[rows])\n            p=(model(x,s).softmax(1)+model(x[:,[0,2,1]],ss).softmax(1)[:,[1,0,2]])*.5\n            result.append(p.numpy())\n        return np.concatenate(result)\n\n    for epoch in range(epochs):\n        started=time.monotonic();model.train();total=0.\n        order=rng.permutation(parts["train"])\n        for start in range(0,len(order),128):\n            rows=order[start:start+128]\n            x=torch.as_tensor(seq[rows].copy(),dtype=torch.long)\n            s=torch.as_tensor(scaled[rows].copy())\n            y=torch.as_tensor(labels[rows].copy(),dtype=torch.long)\n            flip=rng.random(len(rows))<.5\n            x[flip]=x[flip][:,[0,2,1]]\n            s[flip]=torch.as_tensor(scaled_swap[rows[flip]])\n            y[flip]=torch.as_tensor(np.array([1,0,2])[y[flip].numpy()])\n            optimizer.zero_grad(set_to_none=True)\n            loss=loss_fn(model(x,s),y)\n            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            optimizer.step();total+=loss.item()*len(rows)\n        score=log_loss(labels[parts["dev"]],predict(parts["dev"]),labels=[0,1,2])\n        record={"epoch":epoch+1,"train_loss":total/len(order),"dev_log_loss":score,"seconds":time.monotonic()-started}\n        history.append(record);print("RNN",record,flush=True)\n        json_write(out/"rnn_history.json",history)\n        if score<best-.0002:\n            best=score;best_state=copy.deepcopy(model.state_dict());stale=0\n            torch.save({"state_dict":best_state,"vocab":vocab,"max_tokens":max_tokens,"hidden":hidden,\n                        "scaler_mean":scaler.mean_,"scaler_scale":scaler.scale_},out/"rnn_best.pt")\n        else:\n            stale+=1\n            if stale>=3:break\n    model.load_state_dict(best_state);model.eval()\n    predictions=predict(np.arange(len(frame)))\n    embeddings=[]\n    with torch.inference_mode():\n        for j in range(3):\n            chunks=[]\n            for start in range(0,len(frame),256):\n                chunks.append(model.encode(torch.as_tensor(seq[start:start+256,j],dtype=torch.long)).numpy())\n            embeddings.append(np.concatenate(chunks))\n    json_write(out/"rnn_metadata.json",{"vocabulary_size":len(vocab)+2,"max_tokens":max_tokens,\n        "hidden":hidden,"embedding_dimension":hidden*2,"best_dev_log_loss":best,"diagnostics":diagnostics,\n        "epochs_run":len(history),"fit_partition":"train","early_stopping_partition":"dev"})\n    return tuple(embeddings),predictions\n')
print('Bundled arena/rnn.py')

Bundled arena/rnn.py


Bundled source: `arena/vectorizers.py`.

In [5]:
(MODULES / 'arena/vectorizers.py').write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.decomposition import TruncatedSVD\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.preprocessing import normalize\n\nfrom .data import ROOT, SEED, TEXT_COLUMNS, json_write\n\n\ndef tfidf_features(frame, train_idx, out, dimensions=96):\n    """Fit vocabulary, IDF and SVD exclusively on the training partition."""\n    texts = frame[TEXT_COLUMNS].values.T.ravel().tolist()\n    n = len(frame)\n    train_texts = frame.iloc[train_idx][TEXT_COLUMNS].values.T.ravel().tolist()\n    vectorizer = TfidfVectorizer(max_features=40000, ngram_range=(1, 2), min_df=3,\n                                 sublinear_tf=True, strip_accents="unicode", dtype=np.float32)\n    print("Fitting TF-IDF on training texts", flush=True)\n    fit_matrix = vectorizer.fit_transform(train_texts)\n    all_matrix = vectorizer.transform(texts)\n    svd = TruncatedSVD(n_components=dimensions, n_iter=4, random_state=SEED)\n    print("Fitting training-only SVD", flush=True)\n    svd.fit(fit_matrix)\n    dense = normalize(svd.transform(all_matrix)).astype(np.float32)\n    embeddings = tuple(dense[i*n:(i+1)*n] for i in range(3))\n    p, a, b = [all_matrix[i*n:(i+1)*n].tocsr() for i in range(3)]\n    difference = a-b\n    absolute = difference.copy()\n    absolute.data = np.abs(absolute.data)\n    pair = sparse.hstack([p, a, b, difference, absolute], format="csr")\n    swap = sparse.hstack([p, b, a, -difference, absolute], format="csr")\n    out.mkdir(parents=True, exist_ok=True)\n    joblib.dump((vectorizer, svd), out / "tfidf_svd.joblib")\n    json_write(out / "tfidf_metadata.json", {"vocabulary": len(vectorizer.vocabulary_),\n               "dimensions": dimensions, "explained_variance": float(svd.explained_variance_ratio_.sum()),\n               "fit_partition": "train", "ngram_range": [1, 2]})\n    return embeddings, pair, swap\n\n')
print('Bundled arena/vectorizers.py')

Bundled arena/vectorizers.py


Bundled source: `arena/offline.py`.

In [6]:
(MODULES / 'arena/offline.py').write_text('"""Full-data refit for internet-disabled Kaggle inference; no API calls."""\nfrom __future__ import annotations\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport lightgbm as lgb\nfrom scipy import sparse\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom .data import CLASSES,TEXT_COLUMNS,SEED,parse_turns,structural_features,swapped,dense_pair_features,swapped_pair_features\nfrom .models import augment,symmetric_predict\nfrom .rnn import make_sequences,PreferenceRNN\nfrom .vectorizers import tfidf_features\n\n\ndef train_full_rnn(frame,y,stats,epochs):\n    torch.manual_seed(SEED);torch.set_num_threads(4)\n    idx=np.arange(len(y))\n    seq,vocab,_=make_sequences(frame,idx)\n    swapped_stats=structural_features(swapped(frame))\n    scaler=StandardScaler().fit(stats[idx])\n    ss=scaler.transform(stats).astype(np.float32)\n    sw=scaler.transform(swapped_stats).astype(np.float32)\n    model=PreferenceRNN(len(vocab)+2,stats.shape[1])\n    optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)\n    rng=np.random.default_rng(SEED)\n    for epoch in range(epochs):\n        model.train();started=time.monotonic();total=0.\n        order=rng.permutation(idx)\n        for start in range(0,len(order),128):\n            rows=order[start:start+128]\n            x=torch.as_tensor(seq[rows].copy(),dtype=torch.long)\n            s=torch.as_tensor(ss[rows].copy())\n            labels=y[rows].copy()\n            flip=rng.random(len(rows))<.5\n            x[flip]=x[flip][:,[0,2,1]]\n            s[flip]=torch.as_tensor(sw[rows[flip]])\n            labels[flip]=np.array([1,0,2])[labels[flip]]\n            target=torch.as_tensor(labels,dtype=torch.long)\n            optimizer.zero_grad(set_to_none=True)\n            loss=torch.nn.functional.cross_entropy(model(x,s),target,label_smoothing=.02)\n            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step()\n            total+=loss.item()*len(rows)\n        print(f\'Full-data GRU epoch {epoch+1}/{epochs}: train loss {total/len(y):.5f}, {time.monotonic()-started:.1f}s\',flush=True)\n    model.eval();embeddings=[];predictions=[]\n    with torch.inference_mode():\n        for j in range(3):\n            chunks=[]\n            for start in range(0,len(frame),256):\n                chunks.append(model.encode(torch.as_tensor(seq[start:start+256,j],dtype=torch.long)).numpy())\n            embeddings.append(np.concatenate(chunks))\n        for start in range(0,len(frame),256):\n            x=torch.as_tensor(seq[start:start+256],dtype=torch.long)\n            s=torch.as_tensor(ss[start:start+256]);swapstats=torch.as_tensor(sw[start:start+256])\n            predictions.append(((model(x,s).softmax(1)+model(x[:,[0,2,1]],swapstats).softmax(1)[:,[1,0,2]])*.5).numpy())\n    return tuple(embeddings),np.concatenate(predictions)\n\n\ndef train_submission(raw_dir,work_dir,config):\n    raw_dir=Path(raw_dir);work_dir=Path(work_dir);work_dir.mkdir(parents=True,exist_ok=True)\n    train=pd.read_csv(raw_dir/\'train.csv\');test=pd.read_csv(raw_dir/\'test.csv\')\n    y=train[CLASSES].values.argmax(1)\n    for frame in (train,test):\n        for col in TEXT_COLUMNS:frame[col]=frame[col].map(parse_turns)\n    frame=pd.concat([train,test],ignore_index=True);idx=np.arange(len(train))\n    stats=structural_features(frame);swapstats=structural_features(swapped(frame))\n    weights={k:v for k,v in config[\'weights\'].items() if v>0}\n    predictions={};features={}\n    if any(n.startswith(\'tfidf\') for n in weights):\n        embeddings,pair,pairswap=tfidf_features(frame,idx,work_dir)\n        if \'tfidf_sparse__logistic\' in weights:\n            features[\'tfidf_sparse\']=(sparse.hstack([pair,sparse.csr_matrix(stats)],format=\'csr\'),\n                                      sparse.hstack([pairswap,sparse.csr_matrix(swapstats)],format=\'csr\'))\n        features[\'tfidf_svd\']=(dense_pair_features(embeddings,stats),swapped_pair_features(embeddings,stats))\n        del pair,pairswap\n    if any(n.startswith(\'rnn\') for n in weights):\n        embeddings,p=train_full_rnn(frame,y,stats,config[\'rnn_best_epoch\'])\n        predictions[\'rnn__end_to_end\']=p\n        features[\'rnn\']=(dense_pair_features(embeddings,stats),swapped_pair_features(embeddings,stats))\n    features[\'structural\']=(stats,swapstats)\n    for name in weights:\n        if name in predictions:continue\n        representation,kind=name.split(\'__\')\n        x,xswap=features[representation]\n        xx,yy=augment(x,xswap,y,idx)\n        settings=config[\'settings\'][name]\n        if kind==\'logistic\':\n            model=LogisticRegression(C=settings[\'C\'],max_iter=250,solver=\'lbfgs\',random_state=SEED)\n        elif kind==\'gbdt\':\n            model=lgb.LGBMClassifier(n_estimators=settings[\'best_iteration\'],learning_rate=.03,num_leaves=15,\n                min_child_samples=100,colsample_bytree=.8,subsample=.8,subsample_freq=1,reg_lambda=5.,\n                verbosity=-1,n_jobs=4,random_state=SEED,deterministic=True,force_col_wise=True)\n        else:raise ValueError(name)\n        model.fit(xx,yy)\n        predictions[name]=symmetric_predict(model,x,xswap)\n        print(\'Refitted\',name,flush=True)\n        del xx,yy\n    blend=sum(weights[n]*predictions[n][len(train):] for n in weights)\n    logits=np.log(np.clip(blend,1e-12,1))/config[\'temperature\'];logits-=logits.max(1,keepdims=True)\n    p=np.exp(logits);p/=p.sum(1,keepdims=True)\n    submission=pd.DataFrame(p,columns=CLASSES);submission.insert(0,\'id\',test.id.to_numpy())\n    assert submission.id.is_unique and np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1)\n    assert submission.columns.tolist()==pd.read_csv(raw_dir/\'sample_submission.csv\').columns.tolist()\n    submission.to_csv(work_dir/\'submission.csv\',index=False)\n    return submission\n')
print('Bundled arena/offline.py')

Bundled arena/offline.py


In [7]:
CONFIG = {'weights': {'rnn__end_to_end': 0.24792174697553704, 'rnn__gbdt': 0.047036949900340845, 'structural__gbdt': 0.04500826161363299, 'tfidf_sparse__logistic': 0.31729192958648683, 'tfidf_svd__gbdt': 0.3427411119240024}, 'temperature': 0.9049598603951539, 'settings': {'rnn__end_to_end': {}, 'rnn__gbdt': {'best_iteration': 105, 'num_leaves': 15}, 'structural__gbdt': {'best_iteration': 510, 'num_leaves': 15}, 'tfidf_sparse__logistic': {'C': 0.1, 'trials': [{'C': 0.1, 'dev_log_loss': 1.034084677696228}, {'C': 1.0, 'dev_log_loss': 1.1258330345153809}, {'C': 3.0, 'dev_log_loss': 1.0748569965362549}]}, 'tfidf_svd__gbdt': {'best_iteration': 422, 'num_leaves': 15}}, 'rnn_best_epoch': 3}
CONFIG

{'weights': {'rnn__end_to_end': 0.24792174697553704,
  'rnn__gbdt': 0.047036949900340845,
  'structural__gbdt': 0.04500826161363299,
  'tfidf_sparse__logistic': 0.31729192958648683,
  'tfidf_svd__gbdt': 0.3427411119240024},
 'temperature': 0.9049598603951539,
 'settings': {'rnn__end_to_end': {},
  'rnn__gbdt': {'best_iteration': 105, 'num_leaves': 15},
  'structural__gbdt': {'best_iteration': 510, 'num_leaves': 15},
  'tfidf_sparse__logistic': {'C': 0.1,
   'trials': [{'C': 0.1, 'dev_log_loss': 1.034084677696228},
    {'C': 1.0, 'dev_log_loss': 1.1258330345153809},
    {'C': 3.0, 'dev_log_loss': 1.0748569965362549}]},
  'tfidf_svd__gbdt': {'best_iteration': 422, 'num_leaves': 15}},
 'rnn_best_epoch': 3}

In [8]:
from arena.offline import train_submission
submission = train_submission(RAW, WORK, CONFIG)
print('Wrote', WORK / 'submission.csv', 'with', len(submission), 'test rows')
submission.head()


Fitting TF-IDF on training texts


Fitting training-only SVD


Full-data GRU epoch 1/3: train loss 1.05399, 63.4s


Full-data GRU epoch 2/3: train loss 1.03461, 77.1s


Full-data GRU epoch 3/3: train loss 1.02106, 80.0s


Refitted rnn__gbdt


... (2 outputs left out)

Refitted tfidf_sparse__logistic


Refitted tfidf_svd__gbdt


Wrote /home/tarstars/database/llm_classification_finetuning/runs/full_refit/submission.csv with 3 test rows


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.245625,0.311253,0.443122
1,211333,0.408464,0.309312,0.282223
2,1233961,0.231434,0.563340,0.205226
